In [1]:
import copy
import time
import json
import re
import requests
from bs4 import BeautifulSoup

def get_id_list_by_rarity(rarity_code):
    """
    根據稀有度代碼抓取對應的貓咪 ID 列表
    rarity_code 範例: 'n' (基本), 'ex', 'r' (稀有), 'sr' (激稀有), 'ssr' (超激稀有), 'sssr' (傳說稀有)
    """
    url = f"https://battlecats-db.com/unit/status_r_{rarity_code}.html"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')

        ids = []
        for td in soup.find_all('td', string=re.compile(r'^(\d{3})-1$')):
            text = td.text.strip()
            match = re.match(r'^(\d{3})-1$', text)

            if match:
                cat_id = match.group(1)
                if cat_id not in ids:
                    ids.append(cat_id)

        ids.sort()
        return ids

    except requests.exceptions.RequestException as e:
        print(f"[{rarity_code.upper()}] 網頁請求失敗: {e}")
        return []

# ==========================================
# 爬蟲與原始解析模組
# ==========================================
def extract_int(td_node):
    if not td_node: return "NAN"
    soup_tmp = BeautifulSoup(str(td_node), 'html.parser')
    for hidden in soup_tmp.find_all(class_='hide'):
        hidden.decompose()
    text = soup_tmp.get_text().replace(',', '').strip()
    numbers = re.findall(r'\d+', text)
    return int(numbers[0]) if numbers else "NAN"

def extract_normal_list(td_node):
    if not td_node: return "NAN"
    items = [text.strip() for text in td_node.stripped_strings if text.strip()]
    return items if items else "NAN"

def extract_feature_list(td_node, is_instinct=False):
    if not td_node: return "NAN"
    soup_tmp = BeautifulSoup(str(td_node), 'html.parser')
    node = soup_tmp.td if soup_tmp.td else soup_tmp
    for hidden in node.find_all(class_='hide'):
        hidden.decompose()
    for img in node.find_all('img'):
        img.decompose()
    items = []
    if is_instinct:
        for br in node.find_all('br'): br.replace_with(' ')
        for hr in node.find_all('hr'): hr.replace_with('|||SPLIT|||')
    else:
        for br in node.find_all('br'): br.replace_with('|||SPLIT|||')
    full_text = node.get_text()
    blocks = full_text.split('|||SPLIT|||')
    for b in blocks:
        cleaned = re.sub(r'\s+', ' ', b).strip()
        if cleaned and cleaned != "-":
            items.append(cleaned)
    return items if items else "NAN"

def parse_battlecats_html(html_content, cat_id):
    soup = BeautifulSoup(html_content, 'html.parser')
    result = {
        cat_id: {
            "一階的開放條件": "NAN", "一階資料": "NAN", "二階資料": "NAN", "三階資料": "NAN", "四階資料": "NAN"
        }
    }
    form_keys = ["一階資料", "二階資料", "三階資料", "四階資料"]
    for i in range(1, 5):
        form_title = soup.find('td', string=re.compile(rf'No\.{cat_id}-{i}'))
        if not form_title: continue
        form_key = form_keys[i-1]
        name = "NAN"
        name_node = form_title.find_next_sibling('td')
        if name_node: name = name_node.text.strip()

        form_data = {
            "名字": name, "體力": "NAN", "KB": "NAN", "攻擊力": "NAN", "速度": "NAN",
            "DPS": "NAN", "射程": "NAN", "範圍": "NAN", "成本": "NAN",
            "攻擊頻率": "NAN", "攻擊發生": "NAN", "再生產": "NAN",
            "能力": "NAN", "本能": "NAN"
        }
        current_tr = form_title.parent.find_next_sibling('tr')
        while current_tr and not current_tr.find('td', string=re.compile(r'No\.\d{3}-')):
            tds = current_tr.find_all('td', class_='bgc12')
            for td in tds:
                label = td.text.strip()
                val_td = td.find_next_sibling('td')
                if label == "体力": form_data["體力"] = extract_int(val_td)
                elif label == "KB": form_data["KB"] = extract_int(val_td)
                elif label == "攻撃力": form_data["攻擊力"] = extract_int(val_td)
                elif label == "速度": form_data["速度"] = extract_int(val_td)
                elif label == "DPS": form_data["DPS"] = extract_int(val_td)
                elif label == "射程": form_data["射程"] = extract_int(val_td)
                elif label == "コスト": form_data["成本"] = extract_int(val_td)
                elif label == "攻撃頻度F": form_data["攻擊頻率"] = extract_int(val_td)
                elif label == "攻撃発生F": form_data["攻擊發生"] = extract_int(val_td)
                elif label == "再生産F": form_data["再生產"] = extract_int(val_td)
                elif label == "範囲": form_data["範圍"] = val_td.text.strip() if val_td else "NAN"
                elif label == "特性": form_data["能力"] = extract_feature_list(val_td, is_instinct=False)
                elif label == "本能": form_data["本能"] = extract_feature_list(val_td, is_instinct=True)
                elif label == "開放条件":
                    if i == 1: result[cat_id]["一階的開放條件"] = extract_normal_list(val_td)
            current_tr = current_tr.find_next_sibling('tr')
        result[cat_id][form_key] = form_data
    return result

def get_battlecats_data_by_id(cat_id):
    cat_id_str = str(cat_id).zfill(3)
    url = f"https://battlecats-db.com/unit/{cat_id_str}.html"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response.encoding = 'utf-8'
        return parse_battlecats_html(response.text, cat_id_str)
    except requests.exceptions.RequestException as e:
        return {"error": f"網頁請求失敗 (ID: {cat_id_str}): {e}"}

# ==========================================
# 資料庫處理與特徵解析模組
# ==========================================
ABILITY_MAP = {
    "超ダメージ": 1, "極ダメージ": 2, "打たれ強い": 3, "超打たれ強い": 4, "めっぽう強い": 5,
    "渾身の一撃": 6, "クリティカル": 7, "バリアブレイカー": 8, "悪魔シールド貫通": 9,
    "爆波": 14, "小烈波": 13, "烈波": 12, "小波動": 11, "波動": 10,
    "ふっとばす": 15, "動きを止める": 16, "動きを遅くする": 17,
    "遠方範囲全方位攻撃": 19, "全方位攻撃": 19, "遠方範囲攻撃": 18, "遠方攻撃": 18,
    "メタルキラー": 20, "ゾンビキラー": 21, "魂攻撃": 22, "召喚": 23,
    "超生命体特効": 24, "超獣特効": 25, "超賢者特効": 26, "魔女キラー": 27, "使徒キラー": 28,
    "連続攻撃": 29, "1回攻撃": 30, "攻撃力上昇": 31, "攻撃力低下": 32, "1度だけ生き残る": 33,
    "お金x2": 34, "対お城": 35, "メタル": 36, "呪い": 37, "古代の呪い": 37,
    "波動無効": 38, "無効 （波動）": 38, "波動ストッパー": 39,
    "烈波無効": 40, "無効 （烈波）": 40, "爆波無効": 41, "無効 （爆波）": 41,
    "ふっとばす無効": 42, "無効 （ふっとばす）": 42,
    "動きを止める無効": 43, "無効 （動きを止める）": 43, "無効 （止める）": 43,
    "動きを遅くする無効": 44, "無効 （動きを遅くする）": 44, "無効 （遅くする）": 44,
    "攻撃力低下無効": 45, "無効 （攻撃力低下）": 45,
    "ワープ無効": 46, "無効 （ワープ）": 46,
    "古代の呪い無効": 47, "無効 （古代の呪い）": 47, "無効 （呪い）": 47,
    "毒撃無効": 48, "無効 （毒撃）": 48, "攻撃無効": 49
}

def get_default_ability_schema(ability_id):
    """回傳所有能力統一的標準格式"""
    return {
        "能力ID": ability_id, "有此能力": False, "機率": 0, "效果長度": 0,
        "效果強度": 0.0, "最小範圍": 0, "最大範圍": 0, "生效敵人": [False] * 15
    }

def parse_ability_string(text, ability_id):
    """從字串中提取數值並填入標準結構"""
    if "NP" in text:
        return None

    schema = get_default_ability_schema(ability_id)
    schema["有此能力"] = True

    if ability_id in (18, 19):
        match = re.search(r'\((-?\d+)～(-?\d+)\)', text)
        if match:
            schema["最小範圍"] = int(match.group(1))
            schema["最大範圍"] = int(match.group(2))
        return schema

    if ability_id == 23:
        return schema

    if "全ての敵" in text:
        for i in range(10): schema["生效敵人"][i] = True
        exclude_match = re.search(r'（(.*?)除く）', text)
        if exclude_match:
            ex = exclude_match.group(1)
            if "無" in ex: schema["生效敵人"][0] = False
            if "赤" in ex: schema["生效敵人"][1] = False
            if "黒" in ex: schema["生效敵人"][2] = False
            if "浮" in ex: schema["生效敵人"][3] = False
            if "メタル" in ex: schema["生效敵人"][4] = False
            if "天" in ex: schema["生效敵人"][5] = False
            if "エイ" in ex: schema["生效敵人"][6] = False
            if "ゾ" in ex: schema["生效敵人"][7] = False
            if "古" in ex: schema["生效敵人"][8] = False
            if "悪" in ex: schema["生效敵人"][9] = False
            if "魔" in ex: schema["生效敵人"][13] = False
            if "使" in ex: schema["生效敵人"][14] = False
    else:
        if "白い敵" in text or "無属性" in text: schema["生效敵人"][0] = True
        if "赤い敵" in text: schema["生效敵人"][1] = True
        if "黒い敵" in text: schema["生效敵人"][2] = True
        if "浮いてる敵" in text: schema["生效敵人"][3] = True
        if "メタルな敵" in text or ("メタル" in text and ability_id not in [20, 36] and "無効" not in text): schema["生效敵人"][4] = True
        if "天使" in text: schema["生效敵人"][5] = True
        if "エイリアン" in text: schema["生效敵人"][6] = True
        if "ゾンビ" in text: schema["生效敵人"][7] = True
        if "古代種" in text: schema["生效敵人"][8] = True
        if "悪魔" in text: schema["生效敵人"][9] = True
        if "超生命体" in text: schema["生效敵人"][10] = True
        if "超獣" in text: schema["生效敵人"][11] = True
        if "超賢者" in text: schema["生效敵人"][12] = True
        if "魔女" in text: schema["生效敵人"][13] = True
        if "使徒" in text: schema["生效敵人"][14] = True

    prob_match = re.search(r'(?:(\d+)～)?(\d+)％の確率', text)
    if prob_match:
        schema["機率"] = int(prob_match.group(2))
    elif "100％" not in text and any(schema["生效敵人"]):
        schema["機率"] = 100

    dur_match = re.search(r'(?:(\d+)～)?(\d+)F', text)
    if dur_match:
        schema["效果長度"] = int(dur_match.group(2))

    if ability_id == 14:
        mag_match = re.search(r'射程\s*(\d+)', text)
        if mag_match: schema["效果強度"] = float(mag_match.group(1))
    elif ability_id in (10, 11, 12, 13):
        mag_match = re.search(r'Lv\s*(\d+)', text, re.IGNORECASE)
        if mag_match: schema["效果強度"] = float(mag_match.group(1))

    return schema

def build_custom_database(ssr_list):
    db = {}
    for cat_id in ssr_list:
        print(f"正在轉換並解析 ID: {cat_id} ...")
        raw_result = get_battlecats_data_by_id(cat_id)

        if "error" in raw_result:
            continue

        raw_data = raw_result.get(cat_id)
        if not raw_data:
            continue

        processed_cat = {}
        processed_cat["一階的開放條件"] = raw_data.get("一階的開放條件", "NAN")

        def process_normal_form(form_key):
            form_data = raw_data.get(form_key)
            if form_data == "NAN" or not isinstance(form_data, dict): return "NAN"

            new_form = copy.deepcopy(form_data)
            if "本能" in new_form: del new_form["本能"]

            raw_abilities = new_form.get("能力", [])
            if raw_abilities == "NAN": raw_abilities = []

            full_abilities_dict = {str(i): get_default_ability_schema(i) for i in range(1, 50)}
            for text in raw_abilities:
                if "NP" in text: continue
                for key in sorted(ABILITY_MAP.keys(), key=len, reverse=True):
                    if key in text:
                        ab_id = ABILITY_MAP[key]
                        parsed = parse_ability_string(text, ab_id)
                        if parsed:
                            full_abilities_dict[str(ab_id)] = parsed
                        break
            new_form["能力"] = full_abilities_dict
            return new_form

        def process_max_instinct_form(form_key):
            base_form = process_normal_form(form_key)
            if base_form == "NAN": return "NAN"

            raw_form_data = raw_data.get(form_key, {})
            if not isinstance(raw_form_data, dict): return base_form

            instincts = raw_form_data.get("本能", [])
            if instincts == "NAN": return base_form

            for text in instincts:
                if "NP" not in text:
                    continue
                clean_text = re.sub(r'\(.*?NP.*?\)', '', text)

                if "追加" in text:
                    if "対" in text:
                        added_enemies = [False] * 15
                        if "無属性" in clean_text or "白い" in clean_text: added_enemies[0] = True
                        if "赤い" in clean_text: added_enemies[1] = True
                        if "黒い" in clean_text: added_enemies[2] = True
                        if "浮いてる" in clean_text: added_enemies[3] = True
                        if "メタル" in clean_text and "キラー" not in clean_text: added_enemies[4] = True
                        if "天使" in clean_text: added_enemies[5] = True
                        if "エイリアン" in clean_text: added_enemies[6] = True
                        if "ゾンビ" in clean_text: added_enemies[7] = True
                        if "古代種" in clean_text: added_enemies[8] = True
                        if "悪魔" in clean_text: added_enemies[9] = True
                        if "超生命体" in clean_text: added_enemies[10] = True
                        if "超獣" in clean_text: added_enemies[11] = True
                        if "超賢者" in clean_text: added_enemies[12] = True
                        if "魔女" in clean_text: added_enemies[13] = True
                        if "使徒" in clean_text: added_enemies[14] = True

                        for ab_id_str, ab_data in base_form["能力"].items():
                            if ab_data["有此能力"]:
                                for i in range(15):
                                    if added_enemies[i]:
                                        ab_data["生效敵人"][i] = True

                    elif "特性" in text:
                        ab_id = None
                        for key in sorted(ABILITY_MAP.keys(), key=len, reverse=True):
                            if key in clean_text:
                                ab_id = ABILITY_MAP[key]
                                break
                        if ab_id:
                            parsed_ab = parse_ability_string(clean_text, ab_id)
                            if parsed_ab:
                                base_form["能力"][str(ab_id)] = parsed_ab

                elif "強化" in text and "特性" in text:
                    ab_id = None
                    for key in sorted(ABILITY_MAP.keys(), key=len, reverse=True):
                        if key in clean_text:
                            ab_id = ABILITY_MAP[key]
                            break
                    if ab_id:
                        target_ab = base_form["能力"][str(ab_id)]
                        if target_ab["有此能力"]:
                            match_f = re.search(r'～(\d+)F', clean_text)
                            if match_f:
                                target_ab["效果長度"] += int(match_f.group(1))
                            match_p = re.search(r'～(\d+)％', clean_text)
                            if match_p:
                                target_ab["機率"] += int(match_p.group(1))

                elif "上昇" in text:
                    if "基本体力" in text and base_form["體力"] != "NAN":
                        match = re.search(r'～(\d+)％', clean_text)
                        if match:
                            base_form["體力"] = int(base_form["體力"] * (1 + int(match.group(1)) / 100))

                    if "基本攻撃力" in text and base_form["攻擊力"] != "NAN":
                        match = re.search(r'～(\d+)％', clean_text)
                        if match:
                            multiplier = 1 + int(match.group(1)) / 100
                            base_form["攻擊力"] = int(base_form["攻擊力"] * multiplier)
                            if base_form["DPS"] != "NAN":
                                base_form["DPS"] = int(base_form["DPS"] * multiplier)

                elif "減少" in text and "生産コスト" in text and base_form["成本"] != "NAN":
                    match = re.search(r'～(\d+)円', clean_text)
                    if match:
                        base_form["成本"] -= int(match.group(1))

            return base_form

        processed_cat["一階資料"] = process_normal_form("一階資料")
        processed_cat["二階資料"] = process_normal_form("二階資料")
        processed_cat["三階資料"] = process_normal_form("三階資料")
        processed_cat["四階資料"] = process_normal_form("四階資料")

        processed_cat["三階滿本能"] = process_max_instinct_form("三階資料")
        processed_cat["四階滿本能"] = process_max_instinct_form("四階資料")

        # ★ 新增：如果滿本能和原始資料一模一樣（代表沒有本能），就設為 NAN
        if processed_cat["三階滿本能"] == processed_cat["三階資料"]:
            processed_cat["三階滿本能"] = "NAN"

        if processed_cat["四階滿本能"] == processed_cat["四階資料"]:
            processed_cat["四階滿本能"] = "NAN"

        processed_cat["評分"] = "NAN"

        db[cat_id] = processed_cat
        time.sleep(1)

    return db


In [2]:
# ================= 測試與正式執行區塊 =================
if __name__ == "__main__":
    # 定義要抓取的所有稀有度代碼
    rarities = ['n', 'ex', 'r', 'sr', 'ssr', 'sssr']

    # 建立一個終極大字典，用來裝所有稀有度的貓咪
    all_cats_database = {}

    for rarity in rarities:
        print(f"\n========== 開始處理稀有度: {rarity.upper()} ==========")

        # 1. 抓取該稀有度的 ID 列表
        current_ids = get_id_list_by_rarity(rarity)

        if not current_ids:
            print(f"未抓取到 {rarity.upper()} 的 ID，跳過此稀有度。")
            continue

        print(f"總共找到 {len(current_ids)} 隻 {rarity.upper()} 貓咪，準備開始建檔...")

        # 2. 把當前稀有度的 ID 列表丟進去建檔
        rarity_database = build_custom_database(current_ids)

        # 3. 分開儲存：為這個稀有度存一個獨立的 JSON 檔案
        file_name = f'battlecats_{rarity}_db.json'
        with open(file_name, 'w', encoding='utf-8') as f:
            json.dump(rarity_database, f, ensure_ascii=False, indent=2)
        print(f"✅ {rarity.upper()} 解析並建檔完成！已儲存為 {file_name}")

        # 4. 把這次的結果更新到終極大字典中
        all_cats_database.update(rarity_database)

    # 5. 迴圈結束後，把所有貓咪存成一個大檔案
    print("\n========== 所有稀有度處理完畢 ==========")
    with open('battlecats_ALL_db.json', 'w', encoding='utf-8') as f:
        json.dump(all_cats_database, f, ensure_ascii=False, indent=2)

    print(f"🎉 終極資料庫建檔完成！總共收錄了 {len(all_cats_database)} 隻貓咪，請查看 battlecats_ALL_db.json")


========== 開始處理稀有度: N ==========
總共找到 10 隻 N 貓咪，準備開始建檔...
正在轉換並解析 ID: 001 ...
正在轉換並解析 ID: 002 ...
正在轉換並解析 ID: 003 ...
正在轉換並解析 ID: 004 ...
正在轉換並解析 ID: 005 ...
正在轉換並解析 ID: 006 ...
正在轉換並解析 ID: 007 ...
正在轉換並解析 ID: 008 ...
正在轉換並解析 ID: 009 ...
正在轉換並解析 ID: 644 ...
✅ N 解析並建檔完成！已儲存為 battlecats_n_db.json

========== 開始處理稀有度: EX ==========
總共找到 189 隻 EX 貓咪，準備開始建檔...
正在轉換並解析 ID: 010 ...
正在轉換並解析 ID: 011 ...
正在轉換並解析 ID: 012 ...
正在轉換並解析 ID: 013 ...
正在轉換並解析 ID: 014 ...
正在轉換並解析 ID: 015 ...
正在轉換並解析 ID: 016 ...
正在轉換並解析 ID: 017 ...
正在轉換並解析 ID: 018 ...
正在轉換並解析 ID: 019 ...
正在轉換並解析 ID: 020 ...
正在轉換並解析 ID: 021 ...
正在轉換並解析 ID: 022 ...
正在轉換並解析 ID: 023 ...
正在轉換並解析 ID: 024 ...
正在轉換並解析 ID: 025 ...
正在轉換並解析 ID: 026 ...
正在轉換並解析 ID: 027 ...
正在轉換並解析 ID: 028 ...
正在轉換並解析 ID: 029 ...
正在轉換並解析 ID: 030 ...
正在轉換並解析 ID: 046 ...
正在轉換並解析 ID: 055 ...
正在轉換並解析 ID: 063 ...
正在轉換並解析 ID: 083 ...
正在轉換並解析 ID: 102 ...
正在轉換並解析 ID: 103 ...
正在轉換並解析 ID: 104 ...
正在轉換並解析 ID: 109 ...
正在轉換並解析 ID: 121 ...
正在轉換並解析 ID: 122 ...
正在轉換並解析 ID: 124 ...
正

In [3]:
import json
import csv
import os
from collections import defaultdict

# ==========================================
# 1. 讀取 CSV 檔案並處理內部重複 ID 的平均
# ==========================================
csv_file_path = 'score.csv'  # ⚠️ 請確保這裡的檔名與你上傳的一致
csv_scores_lists = defaultdict(list)

print("讀取 CSV 分數表中...")
try:
    with open(csv_file_path, mode='r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        headers = next(reader) # 跳過第一列標題行，避免 KeyError 問題

        for row in reader:
            # 確保該行至少有兩個欄位，且第一欄不是空的
            if len(row) < 2 or not str(row[0]).strip():
                continue

            # 直接使用索引抓取，row[0] 是 ID，row[1] 是分數
            cat_id = str(row[0]).strip().zfill(3)
            try:
                score = float(str(row[1]).strip())
                # 把分數加入該 ID 的專屬列表中
                csv_scores_lists[cat_id].append(score)
            except ValueError:
                continue # 如果分數欄位不是數字就跳過

    # 把收集到的分數列表計算平均
    new_scores = {}
    for cat_id, scores in csv_scores_lists.items():
        avg_score = sum(scores) / len(scores)
        new_scores[cat_id] = round(avg_score, 2)

    print(f"成功讀取並整合了 {len(new_scores)} 隻獨一無二貓咪的新評分資料！\n")

except FileNotFoundError:
    print(f"❌ 找不到 CSV 檔案：{csv_file_path}，請確認檔名。")
    exit()

# ==========================================
# 2. 定義所有需要更新的 JSON 資料庫
# ==========================================
json_files = [
    'battlecats_n_db.json',
    'battlecats_ex_db.json',
    'battlecats_r_db.json',
    'battlecats_sr_db.json',
    'battlecats_ssr_db.json',
    'battlecats_sssr_db.json',
    'battlecats_ALL_db.json'
]

# ==========================================
# 3. 逐一更新 JSON 檔案 (處理與原有分數的平均)
# ==========================================
for json_file in json_files:
    if not os.path.exists(json_file):
        print(f"⚠️ 找不到檔案 {json_file}，跳過。")
        continue

    print(f"🔄 正在處理 {json_file}...")

    with open(json_file, 'r', encoding='utf-8') as f:
        db_data = json.load(f)

    update_count = 0
    average_count = 0

    for cat_id, cat_info in db_data.items():
        if cat_id in new_scores:
            new_score = new_scores[cat_id]
            current_score = cat_info.get("評分", "NAN")

            if current_score == "NAN":
                # JSON 裡是 NAN -> 直接填入 CSV 算好的新分數
                cat_info["評分"] = new_score
                update_count += 1
            else:
                # JSON 裡已經有分數 -> 拿 JSON 舊分數跟 CSV 新分數再平均一次
                try:
                    combined_avg = (float(current_score) + new_score) / 2
                    cat_info["評分"] = round(combined_avg, 2)
                    average_count += 1
                except ValueError:
                    cat_info["評分"] = new_score
                    update_count += 1

    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(db_data, f, ensure_ascii=False, indent=2)

    print(f"  ✅ 完成！直接寫入 {update_count} 筆，與原有分數合併平均 {average_count} 筆。")

print("\n🎉 所有資料庫評分合併大功告成！")

讀取 CSV 分數表中...
❌ 找不到 CSV 檔案：score.csv，請確認檔名。
🔄 正在處理 battlecats_n_db.json...


NameError: name 'new_scores' is not defined

In [ ]:
import json
import os

# ==========================================
# 參數設定
# ==========================================
input_file_path = 'battlecats_ALL_db.json'
training_data_path = 'training_data.json'
predict_data_path = 'data_to_be_predicted.json'

# 初始化兩個用來存放分類資料的字典
training_data = {}
data_to_be_predicted = {}

print("🔄 開始讀取並分類資料庫...")

try:
    # 1. 讀取總資料庫
    with open(input_file_path, 'r', encoding='utf-8') as f:
        all_db = json.load(f)

    train_count = 0
    predict_count = 0

    # 2. 遍歷每隻貓咪的完整資料進行篩選
    for cat_id, cat_info in all_db.items():
        # 取得評分，若欄位不存在預設為 'NAN'
        current_score = cat_info.get("評分", "NAN")

        if current_score == "NAN":
            # 沒有評分，放進待預測名單
            data_to_be_predicted[cat_id] = cat_info
            predict_count += 1
        else:
            # 已有評分，放進訓練集名單
            training_data[cat_id] = cat_info
            train_count += 1

    # 3. 將分類好的資料分別寫入新的 JSON 檔案
    with open(training_data_path, 'w', encoding='utf-8') as f:
        json.dump(training_data, f, ensure_ascii=False, indent=2)

    with open(predict_data_path, 'w', encoding='utf-8') as f:
        json.dump(data_to_be_predicted, f, ensure_ascii=False, indent=2)

    print("\n📊 資料分割報告：")
    print(f"  ✅ 訓練資料集 (已評分) -> 共 {train_count} 筆，已儲存至 {training_data_path}")
    print(f"  🔮 待預測資料集 (無評分) -> 共 {predict_count} 筆，已儲存至 {predict_data_path}")
    print("\n🎉 資料集分割完成！")

except FileNotFoundError:
    print(f"❌ 錯誤：找不到總資料庫檔案「{input_file_path}」，請確認檔案已放置於正確路徑。")
except json.JSONDecodeError:
    print(f"❌ 錯誤：「{input_file_path}」檔案格式損壞，無法正確解析 JSON。")